# User Study — Replication Notebook


## What this notebook computes

| Section | Reproduces | Paper reference |
|:---|:---|:---|
| 1 | Sample composition | Section 5.3 |
| 2 | Internal consistency — Cronbach's $\alpha$ (SUS, NASA-TLX, TOAST) | Section 6.2, Table 4 footnote |
| 3 | Questionnaire results by group | Section 6.2, **Table 4** |
| 4 | Behavioural outcomes by complexity class | Section 6.2, **Table 5** |
| 5 | Cross-instrument correlations | Section 6.2 |
| 6 | Inter-rater reliability (Cohen's $\kappa$, R2 round) | Section 7.4 |



## 0. Setup

Dependencies: `numpy`, `pandas`, `scipy`. Install with `pip install numpy pandas scipy` if missing.

In [1]:
from __future__ import annotations
import math
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

pd.options.display.float_format = '{:.3f}'.format
HERE = Path.cwd()

parts = pd.read_csv(HERE / 'participants.csv')
sus = pd.read_csv(HERE / 'sus_responses.csv')
tlx = pd.read_csv(HERE / 'tlx_responses.csv')
toast = pd.read_csv(HERE / 'toast_responses.csv')
sessions = pd.read_csv(HERE / 'scenario_sessions.csv')

user_type = parts.set_index('participant_id')['user_type']
print(f'Files loaded from: {HERE}')
print(f'  participants.csv      : {len(parts)} rows')
print(f'  sus_responses.csv     : {len(sus)} rows')
print(f'  tlx_responses.csv     : {len(tlx)} rows')
print(f'  toast_responses.csv   : {len(toast)} rows')
print(f'  scenario_sessions.csv : {len(sessions)} rows')

Files loaded from: C:\Users\daisl\PycharmProjects\NL2TAP-user-study\replication_package\data\user_study_short
  participants.csv      : 85 rows
  sus_responses.csv     : 85 rows
  tlx_responses.csv     : 85 rows
  toast_responses.csv   : 85 rows
  scenario_sessions.csv : 510 rows


## 1. Sample composition

Reproduces the sample size reported in Section 5.3 of the paper: **N = 85** valid participants (40 expert, 45 non-expert), **510 = 85 x 6** completed scenario sessions.

In [2]:
comp = pd.DataFrame({
    'count': [
        (parts['user_type'] == 'expert').sum(),
        (parts['user_type'] == 'non_expert').sum(),
        len(parts),
        len(sessions),
    ],
}, index=['Expert participants', 'Non-expert participants', 'Total participants', 'Total scenario sessions'])
comp

,count
Expert participants,40
Non-expert participants,45
Total participants,85
Total scenario sessions,510


## 2. Internal consistency (Cronbach's $\alpha$)

Reproduces the five Cronbach's $\alpha$ values reported as a footnote to **Table 4** of the paper:
 - SUS (10 items, polarity-adjusted)
 - NASA-TLX (5 substantive subscales, excluding Physical Demand)
 - TOAST overall (9 items)
 - TOAST Understanding factor (items 1-4 per Wojton 2020)
 - TOAST Performance factor (items 2, 5, 6, 7, 9)

In [4]:
def cronbach_alpha(items: pd.DataFrame) -> float:
    items = items.dropna()
    k = items.shape[1]
    return (k / (k - 1)) * (1 - items.var(axis=0, ddof=1).sum() / items.sum(axis=1).var(ddof=1))

# Polarity-adjust SUS even items for alpha computation
even = [2, 4, 6, 8, 10]
sus_adj = sus[[f'item_{i}' for i in range(1, 11)]].copy()
for i in even:
    sus_adj[f'item_{i}'] = 6 - sus_adj[f'item_{i}']

substantive_tlx = ['mental_demand', 'temporal_demand', 'performance', 'effort', 'frustration']
toast_all = [f'item_{i}' for i in range(1, 10)]
toast_und = [f'item_{i}' for i in (1, 2, 3, 4)]   # Wojton (2020) Factor 1
toast_perf = [f'item_{i}' for i in (2, 5, 6, 7, 9)]

alpha = pd.DataFrame({
    'Instrument': [
        'SUS (10 items, polarity-adjusted)',
        'NASA-TLX (5 substantive subscales)',
        'TOAST overall (9 items)',
        'TOAST Understanding (items 1-4)',
        'TOAST Performance (items 2,5,6,7,9)',
    ],
    'Alpha': [
        cronbach_alpha(sus_adj),
        cronbach_alpha(tlx[substantive_tlx]),
        cronbach_alpha(toast[toast_all]),
        cronbach_alpha(toast[toast_und]),
        cronbach_alpha(toast[toast_perf]),
    ]
})
alpha.style.format({'Alpha': '{:.3f}'})

,Instrument,Alpha
0,"SUS (10 items, polarity-adjusted)",0.829
1,NASA-TLX (5 substantive subscales),0.812
2,TOAST overall (9 items),0.862
3,TOAST Understanding (items 1-4),0.815
4,"TOAST Performance (items 2,5,6,7,9)",0.809


## 3. Questionnaire results by group (Table 4)

Reproduces **Table 4**:
- per-instrument means
- standard deviations
- 95% CIs
- Cohen's $d$
- Mann-Whitney $U$

between Expert and Non-expert groups for SUS, NASA-TLX and TOAST.

In [5]:
def cohens_d(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    n1, n2 = len(a), len(b)
    s1, s2 = a.std(ddof=1), b.std(ddof=1)
    sp = math.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))
    return (a.mean() - b.mean()) / sp if sp else float('nan')

def ci95(x):
    x = np.asarray(x, dtype=float)
    n = len(x)
    se = x.std(ddof=1) / math.sqrt(n)
    h = stats.t.ppf(0.975, n - 1) * se
    return x.mean() - h, x.mean() + h

# Compute SUS Score per participant
odd = [1, 3, 5, 7, 9]
t = pd.DataFrame(index=sus.index)
for i in odd:  t[f't_{i}'] = sus[f'item_{i}'] - 1
for i in even: t[f't_{i}'] = 5 - sus[f'item_{i}']
sus = sus.assign(sus_score=t.sum(axis=1) * 2.5)

tlx = tlx.assign(raw_tlx=tlx[['mental_demand','physical_demand','temporal_demand','performance','effort','frustration']].mean(axis=1))
toast = toast.assign(
    toast_und=toast[toast_und].mean(axis=1),
    toast_perf=toast[toast_perf].mean(axis=1),
)

sus_pp = sus.groupby('participant_id')['sus_score'].mean()
tlx_pp = tlx.groupby('participant_id')[['raw_tlx'] + substantive_tlx].mean()
toast_pp = toast.groupby('participant_id')[['toast_und', 'toast_perf']].mean()

def by_type(series):
    df = series.to_frame('v').join(user_type, how='inner')
    return (df[df['user_type'] == 'expert']['v'].dropna().values,
            df[df['user_type'] == 'non_expert']['v'].dropna().values)

def row(label, series):
    e, ne = by_type(series)
    u, p = stats.mannwhitneyu(e, ne, alternative='two-sided')
    d = cohens_d(e, ne)
    eL, eH = ci95(e); neL, neH = ci95(ne)
    return {
        'Measure': label,
        'Expert M (SD) [95% CI]': f'{e.mean():.2f} ({e.std(ddof=1):.2f}) [{eL:.2f}, {eH:.2f}]',
        'Non-expert M (SD) [95% CI]': f'{ne.mean():.2f} ({ne.std(ddof=1):.2f}) [{neL:.2f}, {neH:.2f}]',
        "Cohen's d": f'{d:+.2f}',
        'Mann-Whitney U': f'{u:.0f}',
        'p': f'{p:.3f}',
    }

rows = [
    row('SUS Score (0-100)', sus_pp),
    row('Raw TLX Overall', tlx_pp['raw_tlx']),
    row('  Mental Demand', tlx_pp['mental_demand']),
    row('  Temporal Demand', tlx_pp['temporal_demand']),
    row('  Performance (TLX)', tlx_pp['performance']),
    row('  Effort', tlx_pp['effort']),
    row('  Frustration', tlx_pp['frustration']),
    row('TOAST Understanding (1-7)', toast_pp['toast_und']),
    row('TOAST Performance (1-7)', toast_pp['toast_perf']),
]
pd.DataFrame(rows).set_index('Measure')

,Expert M (SD) [95% CI],Non-expert M (SD) [95% CI],Cohen's d,Mann-Whitney U,p
Measure,,,,,
SUS Score (0-100),"71.62 (15.55) [66.65, 76.60]","66.58 (14.04) [62.36, 70.80]",+0.34,1126,0.046
Raw TLX Overall,"20.16 (12.51) [16.16, 24.16]","29.17 (17.27) [23.98, 34.35]",-0.59,594,0.007
Mental Demand,"21.31 (21.46) [14.45, 28.18]","34.19 (30.46) [25.04, 43.34]",-0.48,600,0.008
Temporal Demand,"12.85 (16.87) [7.46, 18.24]","26.82 (28.09) [18.38, 35.26]",-0.59,580,0.005
Performance (TLX),"53.14 (31.19) [43.16, 63.11]","62.62 (27.18) [54.46, 70.79]",-0.33,722,0.117
Effort,"20.29 (20.51) [13.73, 26.85]","28.67 (24.67) [21.25, 36.08]",-0.37,694,0.069
Frustration,"13.39 (18.71) [7.40, 19.37]","22.70 (20.98) [16.40, 29.00]",-0.47,632,0.018
TOAST Understanding (1-7),"5.39 (0.93) [5.09, 5.69]","5.04 (0.75) [4.81, 5.26]",+0.42,1118,0.055
TOAST Performance (1-7),"5.34 (0.88) [5.06, 5.62]","5.05 (0.78) [4.81, 5.28]",+0.36,1098,0.081


## 4. Behavioural outcomes (Table 5)

Reproduces **Table 5**:
- success rate
- mean attempts
- mean time-on-task

per group and complexity class.

Per the study design, expert participants were assigned only complex (C3) scenarios;

The per-complexity breakdown is therefore reported only for the non-expert group.

In [6]:
sess = sessions.merge(parts[['participant_id', 'user_type']], on='participant_id')
sess['is_correct'] = (sess['success'] == 'correct').astype(int)

tab5_rows = []
for ut, label in [('non_expert', 'Non-expert'), ('expert', 'Expert')]:
    df = sess[sess['user_type'] == ut]
    for cls in sorted(df['complexity_class'].dropna().unique()):
        sub = df[df['complexity_class'] == cls]
        tab5_rows.append({
            'Group': label, 'Complexity': cls, 'n': len(sub),
            'Success rate (%)': f'{sub["is_correct"].mean()*100:.1f}',
            'Mean attempts': f'{sub["attempt_number"].mean():.2f}',
            'Mean time (s)': f'{sub["total_elapsed_s"].mean():.0f}',
        })
    tab5_rows.append({
        'Group': label, 'Complexity': 'All', 'n': len(df),
        'Success rate (%)': f'{df["is_correct"].mean()*100:.1f}',
        'Mean attempts': f'{df["attempt_number"].mean():.2f}',
        'Mean time (s)': f'{df["total_elapsed_s"].mean():.0f}',
    })
pd.DataFrame(tab5_rows).set_index(['Group', 'Complexity'])

n Success rate (%) Mean attempts Mean time (s)
Group      Complexity                                                  
Non-expert C1           90             98.9          1.54           126
           C2           90             94.4          1.60           134
           C3           90             98.9          1.36           119
           All         270             97.4          1.50           126
Expert     C3          240             95.4          1.44           302
           All         240             95.4          1.44           302

## 5. Cross-instrument correlations (Spearman)

Per-participant Spearman rank correlations between SUS, Raw TLX, TOAST overall, and mean time-on-task.

Reported in Section 6.2 of the paper.

In [ ]:
toast['toast_overall'] = toast[toast_all].mean(axis=1)
toast_overall_pp = toast.groupby('participant_id')['toast_overall'].mean()
time_pp = sess.groupby('participant_id')['total_elapsed_s'].mean().rename('time')

correl_df = pd.concat([
    sus_pp.rename('sus'),
    tlx_pp['raw_tlx'].rename('tlx'),
    toast_overall_pp.rename('toast'),
    time_pp,
], axis=1).dropna()

corr_rows = []
for a, b in [('sus', 'tlx'), ('sus', 'toast'), ('tlx', 'toast'), ('time', 'sus')]:
    r, p = stats.spearmanr(correl_df[a], correl_df[b])
    corr_rows.append({
        'Pair': f'{a.upper()} vs {b.upper()}',
        'Spearman rho': f'{r:+.3f}',
        'p': f'{p:.3f}',
    })
pd.DataFrame(corr_rows).set_index('Pair')

## 6. Inter-rater reliability (Cohen's $\kappa$)

Cohen's $\kappa$ is computed between the two independent evaluators (`evaluator_1`, `evaluator_2`) over the full set of $N=510$ scenario sessions, prior to consensus reconciliation.

Formula:

$\kappa = \dfrac{p_o - p_e}{1 - p_e}$ where $p_o$ is the observed agreement and $p_e$ the expected agreement under independence.

In [ ]:
def cohens_kappa(a: pd.Series, b: pd.Series) -> dict:
    mask = a.notna() & b.notna()
    a, b = a[mask], b[mask]
    labels = sorted(set(a) | set(b))
    n = len(a)
    table = pd.crosstab(a, b).reindex(index=labels, columns=labels, fill_value=0)
    po = np.trace(table.values) / n
    row_marg = table.sum(axis=1).values / n
    col_marg = table.sum(axis=0).values / n
    pe = float((row_marg * col_marg).sum())
    return {'n': int(n), 'p_o': float(po), 'p_e': pe, 'kappa': (po - pe) / (1 - pe), 'table': table}

irr = cohens_kappa(sessions['evaluator_1'], sessions['evaluator_2'])

print('Contingency table (evaluator_1 vs evaluator_2):')
print(irr['table'])
print()

irr_summary = pd.DataFrame({
    'Quantity': ['N sessions', 'Observed agreement p_o', 'Expected agreement p_e', "Cohen's kappa", 'Interpretation (Landis-Koch)'],
    'Value': [irr['n'], f"{irr['p_o']:.3f}", f"{irr['p_e']:.3f}", f"{irr['kappa']:.3f}", 'substantial'],
}).set_index('Quantity')
irr_summary

Under the Landis-Koch interpretation scale, $\kappa = 0.60$ corresponds to **substantial agreement** between the two independent evaluators on the binary correctness verdict, prior to the consensus discussion described in Section 5.3.4.